In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

In [3]:
train_data = pd.read_csv("dataset/train.csv")
test_data = pd.read_csv("dataset/test.csv")

In [4]:
X = train_data.drop(columns = ['SalePrice', 'Id', 'GarageArea', 'TotRmsAbvGrd'])
y = train_data['SalePrice']
X_test = test_data.drop(columns = ['Id', 'GarageArea', 'TotRmsAbvGrd'])

In [5]:
X_categorical = X.select_dtypes(include = ['object']).columns.tolist()
X_numerical = X.select_dtypes(exclude = ['object']).columns.tolist()

In [6]:
numerical_transformer = SimpleImputer(strategy = 'mean')

In [7]:
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

In [8]:
preprocessor = ColumnTransformer(transformers = [
    ('num', numerical_transformer, X_numerical),
    ('cat', categorical_transformer, X_categorical)
])

In [9]:
param_grid = {
    'decisiontree__max_depth': [None, 10, 20, 30, 40, 50],
    'decisiontree__min_samples_split': [2, 10, 20],
    'decisiontree__min_samples_leaf': [1, 2, 5, 10],
    'decisiontree__max_features': [None, 'sqrt', 'log2']
}

In [10]:
model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('decisiontree', DecisionTreeRegressor(random_state = 42))
])

In [11]:
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error')

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [13]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         SimpleImputer(),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'YearBuilt',
                                                                          'YearRemodAdd',
                                                                          'MasVnrArea',
                                                                          'BsmtFinSF1',
                                                                          'BsmtFinSF2',
                                                                          'BsmtUnfSF',
                                                                          'TotalBsmtSF',
                                                                          '1stFlrSF',
                                                                          '2ndFlrSF',
                                                                          'LowQualFinSF',
                                                                          'GrLivArea',
                                                                          'BsmtFullBath',
                                                                          'BsmtHalfBath...
                                                                          'BsmtFinType2',
                                                                          'Heating',
                                                                          'HeatingQC',
                                                                          'CentralAir',
                                                                          'Electrical', ...])])),
                                       ('decisiontree',
                                        DecisionTreeRegressor(random_state=42))]),
             param_grid={'decisiontree__max_depth': [None, 10, 20, 30, 40, 50],
                         'decisiontree__max_features': [None, 'sqrt', 'log2'],
                         'decisiontree__min_samples_leaf': [1, 2, 5, 10],
                         'decisiontree__min_samples_split': [2, 10, 20]},
             scoring='neg_mean_squared_error')

In [14]:
y_pred = grid_search.predict(X_val)

In [15]:
y_test = grid_search.predict(X_test)

In [16]:
output = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': y_test})
output.to_csv('DecisionTree.csv', index=False)

In [95]:
cv_scores = cross_val_score(grid_search, X, y, cv=5, scoring='neg_mean_squared_error')
print(f'Cross-Validated MSE: {-np.mean(cv_scores)}')
print(r2_score(y_val, y_pred))
rmse = mean_squared_error(y_val, y_pred, squared = False)
print(rmse)
mse = mean_squared_error(y_val, y_pred)
print(f'Mean Squared Error: {mse}')

Cross-Validated MSE: 1464788260.6737149
0.8018759483183089
38983.02330358912
Mean Squared Error: 1519676105.8881724
